# Part 3 — Predict no-shows and choose the model honestly
The test set is used for reporting only: the model, its settings and the threshold are all chosen from the training set.

## Task 3.1 — Features, split and baseline

In [27]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score,
                             f1_score, roc_auc_score)

df = pd.read_csv("appointments_clean.csv")   # row order must not change

X = df[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows", "reminder_sent",
        "booking_channel", "visit_type"]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X columns:", X.shape[1], list(X.columns))
print("Train:", len(X_train), "| Test:", len(X_test))
print("No-shows in test set:", y_test.sum(), f"| overall no-show share: {100*y.mean():.1f}%")
baseline = (y_test == 0).mean()
print(f"Baseline accuracy (always 'Attended'): {baseline:.4f} ({(y_test == 0).sum()} of {len(y_test)})")

X columns: 11 ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Train: 792 | Test: 198
No-shows in test set: 49 | overall no-show share: 22.0%
Baseline accuracy (always 'Attended'): 0.7525 (149 of 198)


## Task 3.2 — Logistic regression

In [28]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)      # fit on the training set only
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
pred = lr.predict(X_test_s)
prob = lr.predict_proba(X_test_s)[:, 1]
print("Accuracy :", round(accuracy_score(y_test, pred), 3))
print("Confusion matrix [[TN, FP], [FN, TP]]:\n", confusion_matrix(y_test, pred))
print("Precision:", round(precision_score(y_test, pred), 3))
print("Recall   :", round(recall_score(y_test, pred), 2))
print("F1       :", round(f1_score(y_test, pred), 2))
print("ROC-AUC  :", round(roc_auc_score(y_test, prob), 3))

coefs = pd.DataFrame({"feature": X.columns, "coef (scaled)": lr.coef_[0], "scale (SD)": scaler.scale_})
print(coefs.round(3).to_string(index=False))

i = list(X.columns).index("reminder_sent")
c = lr.coef_[0][i]
print("\nreminder_sent coefficient (scaled):", round(c, 3))
print("np.exp(coef)          :", round(np.exp(c), 3), "  <- odds ratio per ONE STANDARD DEVIATION of the feature")
print("np.exp(coef / scale)  :", round(np.exp(c / scaler.scale_[i]), 3), "  <- odds ratio for the real No(0) -> Yes(1) change")

Accuracy : 0.788
Confusion matrix [[TN, FP], [FN, TP]]:
 [[139  10]
 [ 32  17]]
Precision: 0.63
Recall   : 0.35
F1       : 0.45
ROC-AUC  : 0.789
                feature  coef (scaled)  scale (SD)
                    age         -0.254      16.656
            distance_km          0.253       6.052
                fee_inr          0.453     184.254
              lead_days          0.614       8.614
         prior_no_shows          0.650       0.969
          reminder_sent         -0.581       0.495
    booking_channel_App         -0.047       0.486
  booking_channel_Phone          0.216       0.481
booking_channel_Walk-In         -0.186       0.435
visit_type_Consultation          0.016       0.493
   visit_type_Follow-up         -0.016       0.493

reminder_sent coefficient (scaled): -0.581
np.exp(coef)          : 0.56   <- odds ratio per ONE STANDARD DEVIATION of the feature
np.exp(coef / scale)  : 0.309   <- odds ratio for the real No(0) -> Yes(1) change


## Task 3.3 — Decision tree and overfitting

In [29]:
for name, tree in [("Full-depth tree", DecisionTreeClassifier(random_state=42)),
                   ("Depth-3 tree   ", DecisionTreeClassifier(random_state=42, max_depth=3))]:
    tree.fit(X_train, y_train)
    tr = accuracy_score(y_train, tree.predict(X_train))
    te = accuracy_score(y_test, tree.predict(X_test))
    print(f"{name}: train acc = {tr:.2f}, test acc = {te:.2f}, depth = {tree.get_depth()}")

Full-depth tree: train acc = 1.00, test acc = 0.67, depth = 15
Depth-3 tree   : train acc = 0.82, test acc = 0.78, depth = 3


## Task 3.4 — Random forest and comparison table

In [30]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_pred = rf.predict(X_test); rf_prob = rf.predict_proba(X_test)[:, 1]
print("RF train accuracy:", round(accuracy_score(y_train, rf.predict(X_train)), 2))
print("RF test accuracy :", round(accuracy_score(y_test, rf_pred), 2))
print("RF recall        :", round(recall_score(y_test, rf_pred), 2))
print("RF ROC-AUC       :", round(roc_auc_score(y_test, rf_prob), 2))
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 3 importances:\n", imp.head(3).round(2))

tree3 = DecisionTreeClassifier(random_state=42, max_depth=3).fit(X_train, y_train)
t3_pred = tree3.predict(X_test); t3_prob = tree3.predict_proba(X_test)[:, 1]
table = pd.DataFrame({
    "model": ["Logistic regression", "Decision tree (depth 3)", "Random forest (100)"],
    "accuracy": [accuracy_score(y_test, pred), accuracy_score(y_test, t3_pred), accuracy_score(y_test, rf_pred)],
    "recall":   [recall_score(y_test, pred), recall_score(y_test, t3_pred), recall_score(y_test, rf_pred)],
    "roc_auc":  [roc_auc_score(y_test, prob), roc_auc_score(y_test, t3_prob), roc_auc_score(y_test, rf_prob)],
}).round(2)
print(table.to_string(index=False))

RF train accuracy: 1.0
RF test accuracy : 0.75
RF recall        : 0.31
RF ROC-AUC       : 0.71
Top 3 importances:
 lead_days      0.22
age            0.20
distance_km    0.19
dtype: float64
                  model  accuracy  recall  roc_auc
    Logistic regression      0.79    0.35     0.79
Decision tree (depth 3)      0.78    0.22     0.70
    Random forest (100)      0.75    0.31     0.71


## Task 3.5 — Tune, choose the model and the threshold, then test once

In [38]:
# (a) tune the forest with cross-validation on the TRAINING set
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}, cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)
print("best_params_:", grid.best_params_)
print("best_score_ (CV ROC-AUC):", round(grid.best_score_, 2))

# (b) choose the final model: 5-fold CV ROC-AUC on the training set only
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
cv = {
    "Logistic regression (pipeline)": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Decision tree (depth 3)": cross_val_score(DecisionTreeClassifier(random_state=42, max_depth=3), X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random forest (100)": cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Tuned random forest": grid.best_score_,
}
cv_table = pd.Series(cv).round(2).sort_values(ascending=False)
print(cv_table)
print("Winner:", cv_table.index[0])

# (c) choose the threshold from out-of-fold TRAINING probabilities
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
rows = []
for t in [0.5, 0.35]:
    f = (oof >= t).astype(int)
    rows.append({"threshold": t, "precision": precision_score(y_train, f), "recall": recall_score(y_train, f),
                 "F1": f1_score(y_train, f), "flagged": int(f.sum())})
thr_table = pd.DataFrame(rows).round(2)
print(thr_table.to_string(index=False))
THRESHOLD = float(thr_table.loc[thr_table["F1"].idxmax(), "threshold"])
print("Chosen threshold (higher F1):", THRESHOLD)

# (d) fit the winner on the training set and evaluate ONCE on the test set
lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]
flag = (probs >= THRESHOLD).astype(int)
print(f"Test set at threshold {THRESHOLD}:")
print("  precision:", round(precision_score(y_test, flag), 2))
print("  recall   :", round(recall_score(y_test, flag), 2))
print("  flagged  :", int(flag.sum()), "of", len(flag))
print("  ROC-AUC  :", round(roc_auc_score(y_test, probs), 2))

# (e) save test_risk_scores.csv
scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": probs.round(4),
    "flag": flag,
})
scores.to_csv("test_risk_scores.csv", index=False)
print(scores.shape, "| rows with flag = 1:", scores["flag"].sum())
print(scores.head())

best_params_: {'max_depth': 3, 'n_estimators': 200}
best_score_ (CV ROC-AUC): 0.77
Logistic regression (pipeline)    0.77
Tuned random forest               0.77
Random forest (100)               0.75
Decision tree (depth 3)           0.68
dtype: float64
Winner: Logistic regression (pipeline)
 threshold  precision  recall   F1  flagged
      0.50       0.58    0.25 0.35       73
      0.35       0.51    0.47 0.49      154
Chosen threshold (higher F1): 0.35
Test set at threshold 0.35:
  precision: 0.57
  recall   : 0.57
  flagged  : 49 of 198
  ROC-AUC  : 0.79
(198, 4) | rows with flag = 1: 49
  appointment_id  no_show  risk_prob  flag
0          A0217        0     0.0216     0
1          A0336        0     0.1509     0
2          A0508        0     0.5187     1
3          A0314        0     0.1913     0
4          A0089        0     0.1677     0
